# NB11d_A9_A10_composite_prepost -- LightGBM-MIA tuning (NB08c_v2 v8 prep chain)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


| pq_id | manifest_key | Baseline AUC |
|---|---|---|
| A9 | `composite_prepost_bands` | 0.786 |
| A10 | `composite_prepost_landuse` | 0.773 |

**Source experiment:** NB08c_v2 v8 (verbatim CONFIG, S0, OOF helpers, LightGBM-MIA classifier, audit_parquet).

**Strategy:** replicate source experiment cell verbatim per parquet; assert baseline AUC within +/- 0.005; run Optuna with PATIENCE=40, RESUME-IF-EXISTS via SQLite; refit best params; save OOF + joblib + JSON + registry entry with `mean_folds_auc`, `std_folds_auc`, `fold_aucs` recorded (publishable cross-city metric).

**Output dir:** `OUTPUTS_DIR / "NB11_V2"` (shared across all NB11 notebooks).

## CELL 1 -- CONFIG (verbatim NB08c_v2 v8 CELL 1) + NB11 additions

In [1]:
# @title CELL 1: NB08c CONFIG
TIER_SELECTION = [0,1,2]
CITY_FILTER = None
CITY_SELECTION = None
REQUIRE_UNOSAT = True
RANDOM_STATE = 42
N_FOLDS = 5

USE_V2 = True

# === NB11 ADDITIONS ===
NB11_NAME = 'NB11d_A9_A10_composite_prepost'
NB11_CELL_ID = 'cell_c3'                # source registry cell_id, preserved

# Per-parquet config: (pq_id, manifest_key, baseline_auc_expected)
NB11_PARQUETS = [
    ('A9', 'composite_prepost_bands', 0.786),
    ('A10', 'composite_prepost_landuse', 0.773),
]
NB11_BASELINE_TOL = 0.005

OPTUNA_BASE_BUDGET = 200
OPTUNA_LOW_AUC_BUDGET = 100
OPTUNA_LOW_AUC_CUTOFF = 0.70
OPTUNA_PATIENCE = 40                       # raised from 20 -- TPE needs room to explore beyond baseline
OPTUNA_DIRECTION = 'maximize'
OPTUNA_SEED = 42

# === THREAD BUDGET (avoid oversubscription when running parallel notebooks) ===
import os as _os, multiprocessing as _mp
NB11_PARALLEL_NOTEBOOKS = 1                                  # set to N if running N notebooks concurrently
NB11_CORES_TOTAL = _mp.cpu_count()
NB11_THREADS_PER_NB = max(1, NB11_CORES_TOTAL // NB11_PARALLEL_NOTEBOOKS)
# Env vars must be set BEFORE numpy/BLAS imports (i.e. before CELL 2 global_setup).
# Run with a fresh kernel for these to take effect at numpy import time.
_os.environ['OMP_NUM_THREADS']      = str(NB11_THREADS_PER_NB)
_os.environ['MKL_NUM_THREADS']      = str(NB11_THREADS_PER_NB)
_os.environ['OPENBLAS_NUM_THREADS'] = str(NB11_THREADS_PER_NB)
_os.environ['NUMEXPR_NUM_THREADS']  = str(NB11_THREADS_PER_NB)
_os.environ['BLIS_NUM_THREADS']     = str(NB11_THREADS_PER_NB)
print(f"[threads] cores={NB11_CORES_TOTAL}  parallel_nbs={NB11_PARALLEL_NOTEBOOKS}  "
      f"threads_per_nb={NB11_THREADS_PER_NB}")


[threads] cores=24  parallel_nbs=1  threads_per_nb=24


## CELL 1b -- enforce thread cap dynamically (BLAS/OpenMP)

Belt-and-suspenders to the env vars: if numpy/BLAS were imported on an earlier kernel state with no env-var limit, `threadpoolctl` clamps the active pools NOW without a kernel restart. Without this, LightGBM/XGBoost's OpenMP and sklearn's BLAS still spawn 24 threads each.

Set `NB11_PARALLEL_NOTEBOOKS` in CELL 1 to match the number of NB11 notebooks you'll run concurrently:
- 1 notebook  -> 24 threads per nb (no contention)
- 2 notebooks -> 12 threads each
- 3 notebooks ->  8 threads each
- 4 notebooks ->  6 threads each


In [2]:
# @title CELL 1b: dynamic thread cap via threadpoolctl
try:
    from threadpoolctl import threadpool_limits, threadpool_info
    _tp_limit = threadpool_limits(limits=NB11_THREADS_PER_NB)
    print(f"[threadpoolctl] limited BLAS/OpenMP pools to {NB11_THREADS_PER_NB} threads")
    for info in threadpool_info():
        print(f"  {info.get('prefix'):20s} ({info.get('user_api')})  "
              f"current_threads={info.get('num_threads')}  max={info.get('num_threads')}")
except ImportError:
    print("[threadpoolctl] NOT INSTALLED -- run: pip install threadpoolctl")
    print("                env vars only; may not affect pools loaded before this cell.")


[threadpoolctl] limited BLAS/OpenMP pools to 24 threads


## CELL 2 -- LOAD GLOBAL SETUP (verbatim NB08c_v2 v8)

In [3]:
# @title CELL 2: LOAD GLOBAL SETUP
import platform, os, json
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-06-21 17:13:15
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: True
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     934.9/7452.0 GB (6517.1 GB free)
  GDrive (F:)     1406.2/3726.0 GB (2319.8 GB free)
  Local data      11564.8/14901.9 GB (3337.1 GB free)
  Data stack      1406.2/3726.0 GB (2319.8 GB free)
  WSL ext4        69.0/1006.9 GB (886.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

## CELL S0 -- MANIFEST + BUILDINGS + SHARED HELPERS (verbatim NB08c_v2 v8)

In [4]:
# @title CELL S0: LOAD MANIFEST + BUILDINGS + SHARED HELPERS
import sys, importlib, re, gc, time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score)
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL S0: NB08c v2 - LOAD MANIFEST + BUILDINGS + SHARED HELPERS")
print("=" * 70)

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import metadata_filter
importlib.reload(metadata_filter)
from metadata_filter import (
    ID_COLS, LABEL_COLS, META_COLS_SET,
    is_metadata, is_non_feature,
    select_feature_columns, build_role_overrides,
)

# --- load v2 manifest -------------------------------------------------
MANIFEST_V2_PATH = DATASET_ROOT_V2 / 'parquet_manifest.json'
if not MANIFEST_V2_PATH.exists():
    raise FileNotFoundError(f"V2 manifest not found: {MANIFEST_V2_PATH}")
with open(MANIFEST_V2_PATH) as _f:
    MANIFEST = json.load(_f)
print(f"  Manifest: {MANIFEST_V2_PATH}")
print(f"  Version: {MANIFEST['version']}  Created: {MANIFEST['created']}")
print(f"  Parquets in manifest: {len(MANIFEST['parquets'])}")

# --- load building metadata (v2 path) --------------------------------
_tiers = TIER_SELECTION if isinstance(TIER_SELECTION, list) else [0, 1, 2]
_bldg_pattern = str(DATASET_ROOT_V2 / 'bda_buildings_t{tier}.parquet')
df_bldg = load_tier_parquets(_bldg_pattern, _tiers)
CITIES_TO_PROCESS, _battle_dates = resolve_cities(
    tier_selection=TIER_SELECTION,
    city_selection=[CITY_FILTER] if isinstance(CITY_FILTER, str) else CITY_FILTER,
    require_unosat=REQUIRE_UNOSAT,
)
df_bldg = df_bldg[df_bldg['city'].isin(CITIES_TO_PROCESS)].copy()
df_bldg = df_bldg[df_bldg['damage_binary'] >= 0].copy()
print(f"  Cities: {len(CITIES_TO_PROCESS)}")
print(f"  Buildings: {len(df_bldg)} (damaged={int((df_bldg['damage_binary']==1).sum())}, "
      f"undamaged={int((df_bldg['damage_binary']==0).sum())})")

TARGET_COL = 'damage_binary'
AUDIT_LOG = {}

# ---- NB12 overlap-analysis: canonical OOF predictions schema ----
import hashlib as _hashlib, datetime as _dt
EXPERIMENT_ID = _dt.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + _hashlib.md5(
    f'nb08c_{TIER_SELECTION}_{REQUIRE_UNOSAT}'.encode()).hexdigest()[:6]
print(f"  EXPERIMENT_ID: {EXPERIMENT_ID}")

def save_oof_from_result(res, model_id, oof_dir, variant_id='',
                         is_final=False, threshold=0.5):
    '''Canonical OOF schema: building_id, city, fold_id, y_true, y_proba, y_pred,
       cm_class, model_id, experiment_id, variant_id, is_final.'''
    if res is None:
        return None
    needed = ('y_true', 'y_proba', 'groups', 'building_id', 'fold_id')
    if not all(k in res for k in needed):
        print(f"    [save_oof] skipped {model_id}: missing keys (need {needed})")
        return None
    y_proba = np.asarray(res['y_proba'])
    yt = np.asarray(res['y_true']).astype(int)
    fold = np.asarray(res['fold_id']).astype(int)
    bid = np.asarray(res['building_id'])
    cty = np.asarray(res['groups'])
    yp = (y_proba >= threshold).astype(int)
    cm_class = np.where(yt == 1,
                        np.where(yp == 1, 'TP', 'FN'),
                        np.where(yp == 1, 'FP', 'TN'))
    oof_df = pd.DataFrame({
        'building_id':   bid,
        'city':          cty,
        'fold_id':       fold,
        'y_true':        yt,
        'y_proba':       y_proba.astype(float),
        'y_pred':        yp,
        'cm_class':      cm_class,
        'model_id':      model_id,
        'experiment_id': EXPERIMENT_ID,
        'variant_id':    variant_id,
        'is_final':      bool(is_final),
    })
    oof_dir = Path(oof_dir)
    oof_dir.mkdir(parents=True, exist_ok=True)
    out = oof_dir / f'oof_{model_id}__{EXPERIMENT_ID}.parquet'
    oof_df.to_parquet(out, index=False)
    n_tp = int((cm_class == 'TP').sum()); n_fn = int((cm_class == 'FN').sum())
    n_fp = int((cm_class == 'FP').sum()); n_tn = int((cm_class == 'TN').sum())
    print(f"    saved oof -> {out.name}  TP={n_tp} FN={n_fn} FP={n_fp} TN={n_tn}")
    return out

# --- leakage filter ---------------------------------------------------
LEAKAGE_EXCLUDE_PATTERNS = [
    r's1__coh__scenes_observed',
    r's1__vv__scenes_observed',
    r's2__scenes_observed',
    r's2__lu__scenes_observed',
    r's2__obs_count__',
    r's2__qa__cloud_freq__',
    r's2__visibility__cloud__freq',
    r'was_observed_',
]
# -- leaky filter (legacy NB08c behavior, keeps was_observed_ flags; used by C10 for delta comparison) --
LEAKAGE_LEAKY_PATTERNS = [
    r's1__coh__scenes_observed',
    r's1__vv__scenes_observed',
    r's2__scenes_observed',
    r's2__lu__scenes_observed',
    r's2__obs_count__',
    r's2__qa__cloud_freq__',
    r's2__visibility__cloud__freq',
]
_LEAKAGE_LEAKY_RE = [re.compile(p) for p in LEAKAGE_LEAKY_PATTERNS]

def drop_leakage_leaky(cols):
    '''LEGACY filter that keeps was_observed_* flags. For C10 delta only. Do not use elsewhere.'''
    return [c for c in cols if not any(rx.search(c) for rx in _LEAKAGE_LEAKY_RE)]
_LEAKAGE_RE = [re.compile(p) for p in LEAKAGE_EXCLUDE_PATTERNS]

def drop_leakage(cols):
    return [c for c in cols if not any(rx.search(c) for rx in _LEAKAGE_RE)]

def _has_count_stat(c):
    parts = c.split('__')
    if len(parts) < 4:
        return False
    return parts[3].split('_')[0] == 'count'

# --- v2 manifest-driven loader (copy of NB09a v2 signature) ----------
def load_v2(manifest_key, tiers=None, columns=None, sample_frac=None):
    '''Load a v2 parquet by manifest key, merge with buildings, return df.'''
    if manifest_key not in MANIFEST['parquets']:
        raise KeyError(f"Manifest key not found: {manifest_key}")
    info = MANIFEST['parquets'][manifest_key]
    fname_tmpl = Path(info['pattern']).name
    tiers = tiers if tiers is not None else _tiers

    frames = []
    for t in tiers:
        pf = DATASET_ROOT_V2 / fname_tmpl.format(tier=t)
        if not pf.exists():
            continue
        if columns is not None:
            join_cols = info.get('join_keys', ['city', 'building_id'])
            read_cols = list(dict.fromkeys(join_cols + list(columns)))
            try:
                df_t = pd.read_parquet(pf, columns=read_cols)
            except Exception:
                df_t = pd.read_parquet(pf)
                keep = [c for c in read_cols if c in df_t.columns]
                df_t = df_t[keep]
        else:
            df_t = pd.read_parquet(pf)
        if sample_frac is not None and 0 < sample_frac < 1:
            df_t = df_t.sample(frac=sample_frac, random_state=RANDOM_STATE)
        frames.append(df_t)
    if not frames:
        raise FileNotFoundError(f"No v2 parquets found for key={manifest_key} tiers={tiers}")
    df = pd.concat(frames, ignore_index=True)
    del frames

    df = df.loc[:, ~df.columns.duplicated()]
    df = df[df['city'].isin(CITIES_TO_PROCESS)].copy()
    if TARGET_COL not in df.columns:
        df = df.merge(df_bldg[['building_id', 'city', TARGET_COL]].drop_duplicates(),
                      on=['building_id', 'city'], how='inner')
    df = df[df[TARGET_COL] >= 0].copy()
    df = df.loc[:, ~df.columns.duplicated()]
    return df

# --- v8: direction-aware temporal aggregation (matched-filter principle) ---
# Damage at the temporal axis is a sparse-event spike. Per Kay 1998 (Detection
# Theory) and Glaz-Naus-Wallenstein 2001 (Scan Statistics), the rolling extremum
# is the Neyman-Pearson optimal detector for a 1-of-N spike on stationary noise.
# v7 universal mean+std attenuated the spike; v8 dispatches per modality direction.
# Reference: DECISION_Rolling_Block_Accumulators.md Sec 5; DECISION_NB05b_v29 Sec 1.

DIRECTION_BY_MODALITY = {
    # SAR drop signals (damage -> coherence loss, VV backscatter loss on rough->smooth or rubble)
    'coh_vv': 'drop', 'coh_vh': 'drop',
    'vv': 'drop',
    # SAR mixed-sign (volume scattering can go either direction with rubble)
    'vh': 'mixed',
    # MS visible bands -- no clean damage direction (illumination + atmosphere dominate)
    'b02': 'unknown', 'b03': 'unknown', 'b04': 'unknown',
    'b05': 'unknown', 'b07': 'unknown',
    # MS NIR -- vegetation loss after damage drives NIR drop
    'b08': 'drop', 'b8a': 'drop',
    # MS SWIR -- rubble / bare soil exposure drives SWIR rise (Aimaiti 2022)
    'b11': 'rise', 'b12': 'rise',
    # MS derived indices
    'nbr':   'mixed',  # burn / vegetation loss / rubble (sign depends on land cover)
    'ndvi':  'drop',   # vegetation loss
    'savi':  'drop',   # soil-adjusted vegetation
    'bsi':   'rise',   # bare-soil exposure rises with rubble
    'mndwi': 'mixed',  # water -- flood-rise vs drying-drop
    'ndbi':  'drop',   # built-up index drops with built-up loss
    'ndsi':  'mixed',  # snow seasonal noise, no damage direction
    'ibi':   'drop',   # built-up index variant
    'baei':  'drop',   # built-up area extraction index
    'ui':    'drop',   # urban index
}

# Manifest keys whose long parquets are already pixel-temporal-extremized in NB03e R2b.
# For these, mean+std-over-time preserves the per-rolling-window pattern; a second
# temporal extremum would collapse to the full-post-battle accumulators (A14/A19/A20/A21),
# making A23/A24/A25 redundant.
ACCUM_LONG_KEYS = {'rolling_accum_coh', 'rolling_accum_card', 'rolling_accum_ms'}

def _parse_modality_token(col):
    '''Return the modality token (second `__`-separated piece) of a feature column.
       Examples:
         s2__b02__mean              -> b02
         s1__coh_vv__zscore__mean   -> coh_vv
         s1__vv__roll7__min         -> vv
         s2__landuse__urban_frac    -> landuse  (returns 'landuse', falls into 'unknown')
       Returns None if the column has fewer than 2 `__`-separated pieces.'''
    parts = col.split('__')
    if len(parts) < 2:
        return None
    return parts[1].lower()

def _temporal_reductions_for_column(col):
    '''Return the list of pandas agg names to apply over the date axis for `col`,
       chosen by modality direction:
         drop    -> [min, mean, std]
         rise    -> [max, mean, std]
         mixed   -> [min, max, mean, std]
         unknown -> [mean, std]   (legacy v7 fallback for non-mappable columns)
       The `mean` and `std` slots are kept in every direction so v7 features are
       a strict subset of v8 features (additive change, no information lost).'''
    modality = _parse_modality_token(col)
    direction = DIRECTION_BY_MODALITY.get(modality, 'unknown')
    if direction == 'drop':
        return ['min', 'mean', 'std']
    if direction == 'rise':
        return ['max', 'mean', 'std']
    if direction == 'mixed':
        return ['min', 'max', 'mean', 'std']
    return ['mean', 'std']

# --- long-format aggregation (v8: direction-aware temporal collapse) ---------
def aggregate_long_to_wide(df, feat_cols, manifest_key=None):
    '''Collapse a long-format per-scene parquet to per-building summary.

    v8: direction-aware temporal aggregation per matched-filter principle.
    - manifest_key in ACCUM_LONG_KEYS  -> universal mean+std (already extremized in R2b)
    - otherwise                        -> per-column dispatch via _temporal_reductions_for_column

    Backward compatible call: `aggregate_long_to_wide(df, feat_cols)` (manifest_key=None)
    falls through to per-column dispatch with v7-equivalent fallback for unknown columns.
    '''
    feat_cols = [c for c in feat_cols if c in df.columns]
    if not feat_cols:
        return (df.groupby(['city', 'building_id'], sort=False, observed=True)
                  .size().reset_index()[['city', 'building_id']])

    if manifest_key in ACCUM_LONG_KEYS:
        agg_dict = {c: ['mean', 'std'] for c in feat_cols}
    else:
        agg_dict = {c: _temporal_reductions_for_column(c) for c in feat_cols}

    agg = df.groupby(['city', 'building_id'], sort=False, observed=True).agg(agg_dict)
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    agg = agg.reset_index()
    return agg

# --- manifest-derived feature set for a given parquet ----------------
def get_parquet_features(manifest_key, df):
    '''Return the cleaned feature column list for a parquet: manifest features,
       intersected with df columns, minus leakage, minus count stats, minus all-NaN.

       Uses metadata_filter.select_feature_columns() so that id/label/metadata
       columns (including any was_observed_* flag, regardless of suffix) are
       excluded centrally rather than by a locally-maintained list.'''
    info = MANIFEST['parquets'][manifest_key]
    raw = [c for c in info.get('feature_columns', []) if c in df.columns]
    if not raw:
        # fallback: metadata_filter picks numeric features, excluding id/label/meta
        # and any was_observed_* via the centralized pattern rule.
        raw = select_feature_columns(df)
    else:
        # manifest-declared features: still scrub in case the manifest lists a
        # column that is actually metadata (defensive, typically a no-op).
        raw = [c for c in raw if not is_non_feature(c)]
    clean = drop_leakage(raw)
    clean = [c for c in clean if not _has_count_stat(c)]
    nan_rate = df[clean].isna().mean() if clean else pd.Series(dtype=float)
    clean = [c for c in clean if nan_rate.get(c, 1.0) < 1.0]
    return clean

# --- LightGBM feature prep (no imputation, native NaN) --------------
def prepare_features_mia(df, feat_cols):
    '''Return X (with NaN preserved), y, groups, building_ids, clean_cols.'''
    if not feat_cols:
        return None
    X = df[feat_cols].to_numpy(dtype=np.float32, copy=False)
    y = df[TARGET_COL].values
    groups = df['city'].values
    building_ids = df['building_id'].values
    return X, y, groups, building_ids, feat_cols

# --- single-protocol GroupKFold evaluator ---------------------------
def evaluate_audit(clf, X, y, groups, experiment_name, n_folds=N_FOLDS,
                   building_ids=None):
    n_cities = len(np.unique(groups))
    n_folds_actual = min(n_folds, n_cities)
    if n_folds_actual < 2:
        print(f"    SKIP {experiment_name}: only {n_cities} cities")
        return None
    gkf = GroupKFold(n_splits=n_folds_actual)
    y_proba_oof = np.full(len(y), np.nan)
    fold_id = np.full(len(y), -1, dtype=int)
    fold_aucs = []
    for fold_idx, (tr, te) in enumerate(gkf.split(X, y, groups)):
        clf_c = clone(clf)
        clf_c.fit(X[tr], y[tr])
        proba = clf_c.predict_proba(X[te])[:, 1]
        y_proba_oof[te] = proba
        fold_id[te] = fold_idx
        if len(np.unique(y[te])) > 1:
            fold_aucs.append(roc_auc_score(y[te], proba))
    valid = ~np.isnan(y_proba_oof)
    y_v, p_v = y[valid], y_proba_oof[valid]
    pred_v = (p_v >= 0.5).astype(int)
    bid_v = np.asarray(building_ids)[valid] if building_ids is not None else np.arange(len(y))[valid].astype(str)
    result = {
        'experiment': experiment_name,
        'auc_groupkfold': roc_auc_score(y_v, p_v),
        'f1_groupkfold': f1_score(y_v, pred_v),
        'precision': precision_score(y_v, pred_v, zero_division=0),
        'recall': recall_score(y_v, pred_v, zero_division=0),
        'fold_aucs': fold_aucs,
        'auc_mean': float(np.mean(fold_aucs)) if fold_aucs else float('nan'),
        'auc_std': float(np.std(fold_aucs)) if fold_aucs else float('nan'),
        'n_features': X.shape[1], 'n_buildings': int(valid.sum()),
        'n_cities': int(len(np.unique(groups))),
        'y_true': y_v, 'y_proba': p_v, 'groups': groups[valid],
        'building_id': bid_v, 'fold_id': fold_id[valid],
    }
    AUDIT_LOG[experiment_name] = result
    print(f"    {experiment_name:40s} AUC={result['auc_groupkfold']:.3f}  "
          f"F1={result['f1_groupkfold']:.3f}  n={result['n_buildings']:,}  "
          f"cities={result['n_cities']}  nfeat={result['n_features']}")
    return result

# --- output paths + registry ----------------------------------------
import matplotlib.pyplot as plt
from datetime import datetime as _dt

OUT_DIR = OUTPUTS_DIR / "NB08c_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_result(data, name, cell_id, fmt='csv'):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    if fmt == 'csv' and isinstance(data, pd.DataFrame):
        path = cell_dir / f"{name}_{ts}.csv"
        data.to_csv(path, index=False)
    elif fmt == 'json':
        path = cell_dir / f"{name}_{ts}.json"
        with open(path, 'w') as fh:
            json.dump(data, fh, indent=2, default=str)
    else:
        raise ValueError(f"Unknown fmt={fmt}")
    print(f"  Saved: {path.relative_to(OUT_DIR)} ({path.stat().st_size / 1024:.1f} KB)")
    return path

def save_fig(fig, name, cell_id, dpi=150):
    cell_dir = OUT_DIR / cell_id
    cell_dir.mkdir(parents=True, exist_ok=True)
    ts = _dt.now().strftime('%Y%m%d_%H%M%S')
    path = cell_dir / f"{name}_{ts}.png"
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  Plot: {path.relative_to(OUT_DIR)}")
    return path

from bda_results import ResultRegistry
registry = ResultRegistry(RESULTS_ROOT, notebook='NB08c_v2')

def log_audit(res, cell_id, manifest_key, note=''):
    if res is None:
        return
    registry.log_experiment(
        cell_id=cell_id, experiment_name=res['experiment'],
        parquet_name=f'bda_{manifest_key}_v2', feature_set_name='manifest_default_minus_leakage',
        classifier_name='LightGBM-300-MIA', classifier_params=LGBM_PARAMS,
        feature_cols=[], cv_method='GroupKFold', n_folds=N_FOLDS,
        imputation='native_nan', y_true=res['y_true'], y_proba=res['y_proba'],
        groups=res['groups'], note=note, tags=['nb08c_audit'],
    )

print(f"  Helpers: load_v2(), get_parquet_features(), aggregate_long_to_wide(), evaluate_audit()")
print(f"  Output:  {OUT_DIR}")
print(f"  Registry: NB08c_v2")


CELL S0: NB08c v2 - LOAD MANIFEST + BUILDINGS + SHARED HELPERS
  Manifest: /mnt/f/PROJECTS/masterthesis/data_stack/dataset/V2/parquet_manifest.json
  Version: v2  Created: 2026-04-26T16:31:26.125203
  Parquets in manifest: 37
  load_tier_parquets: 3 tiers, 907371 rows
  Cities: 21
  Buildings: 598595 (damaged=7332, undamaged=591263)
  EXPERIMENT_ID: 20260621_171446_115af1
  ResultRegistry: /content/drive_f/masterthesis/results/registry (run_id=20260621_171446)
  Helpers: load_v2(), get_parquet_features(), aggregate_long_to_wide(), evaluate_audit()
  Output:  /content/drive_f/masterthesis/data/outputs/NB08c_v2
  Registry: NB08c_v2


## CELL S0b -- OOF PLOT + SUMMARY HELPERS (verbatim NB08c_v2 v8)

In [5]:
# @title CELL S0b: OOF PLOT + SUMMARY HELPERS
# TP=red, TN=green, FP=pink, FN=gold
import matplotlib.pyplot as plt

CM_COLORS = {'TP': '#d62728', 'TN': '#2ca02c', 'FP': '#ff9ecb', 'FN': '#ffd700'}
CM_LABELS = {'TP': 'Destroyed (TP)', 'TN': 'Not destroyed (TN)',
             'FP': 'False positive (FP)', 'FN': 'False negative (FN)'}

# alias df_bldg to df_buildings for plotting helper compatibility
df_buildings = df_bldg

def print_cm_summary(oof_path_or_df):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else '(unknown)'
    print(f"  model_id: {model_id}   n={len(oof)}")
    overall = oof['cm_class'].value_counts()
    for cls in ['TP', 'TN', 'FP', 'FN']:
        n = int(overall.get(cls, 0))
        pct = 100.0 * n / len(oof) if len(oof) else 0
        print(f"    {cls:3s} {CM_LABELS[cls]:28s}  n={n:6d}  ({pct:5.1f}%)")
    by_city = (oof.groupby('city')['cm_class']
                 .value_counts().unstack(fill_value=0))
    for cls in ['TP', 'TN', 'FP', 'FN']:
        if cls not in by_city.columns:
            by_city[cls] = 0
    by_city = by_city[['TP', 'TN', 'FP', 'FN']]
    by_city['recall']    = by_city['TP'] / (by_city['TP'] + by_city['FN']).replace(0, np.nan)
    by_city['precision'] = by_city['TP'] / (by_city['TP'] + by_city['FP']).replace(0, np.nan)
    print(f"\n  Per-city:")
    print(by_city.to_string(float_format=lambda x: f'{x:.3f}' if pd.notna(x) else '-'))
    return by_city

def plot_cm_spatial(oof_path_or_df, save_name=None, figsize=(14, 10),
                    buildings_df=None, point_size=4):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    bdf = buildings_df if buildings_df is not None else df_buildings
    if 'centroid_x' not in bdf.columns or 'centroid_y' not in bdf.columns:
        print("  plot_cm_spatial: no centroid_x/centroid_y in buildings_df")
        return None
    plot_df = oof.merge(bdf[['building_id', 'city', 'centroid_x', 'centroid_y']],
                        on=['building_id', 'city'], how='inner')
    if len(plot_df) == 0:
        print("  plot_cm_spatial: no building matches")
        return None
    cities = sorted(plot_df['city'].unique())
    ncols = min(3, len(cities))
    nrows = (len(cities) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else ''
    fig.suptitle(f'Confusion-matrix map: {model_id}', fontsize=11)
    draw_order = ['TN', 'FP', 'FN', 'TP']
    for i, city in enumerate(cities):
        ax = axes[i // ncols][i % ncols]
        cdf = plot_df[plot_df['city'] == city]
        for cls in draw_order:
            pts = cdf[cdf['cm_class'] == cls]
            if len(pts) == 0:
                continue
            ax.scatter(pts['centroid_x'], pts['centroid_y'],
                       c=CM_COLORS[cls], s=point_size, alpha=0.75,
                       edgecolors='none',
                       label=f"{CM_LABELS[cls]} (n={len(pts)})")
        ax.set_title(f"{city}  (n={len(cdf)})", fontsize=9)
        ax.set_aspect('equal')
        ax.legend(loc='best', fontsize=6, markerscale=2, framealpha=0.85)
        ax.tick_params(labelsize=7)
    for j in range(len(cities), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if save_name:
        fig.savefig(OUT_DIR / f'{save_name}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    return fig

def plot_cm_oof_all(oof_dir, skip_variants=()):
    d = Path(oof_dir)
    files = sorted(d.glob('oof_*.parquet'))
    print(f"  Found {len(files)} OOF parquets in {d}")
    for p in files:
        oof = pd.read_parquet(p)
        v = oof['variant_id'].iloc[0] if 'variant_id' in oof.columns else ''
        if any(sv in v for sv in skip_variants):
            print(f"  skip {p.name}  (variant_id={v})")
            continue
        print(f"\n  {p.name}")
        print_cm_summary(oof)
        plot_cm_spatial(oof, save_name=p.stem)


## CELL H -- CLASSIFIER (verbatim LightGBM-MIA from NB08c_v2 v8)

In [6]:
# @title CELL H: CLASSIFIER (LightGBM native NaN)
try:
    from lightgbm import LGBMClassifier
except ImportError:
    raise ImportError("LightGBM required for NB08c single-protocol audit. pip install lightgbm")

LGBM_PARAMS = {
    'n_estimators': 150,
    'max_depth': 6,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1,
    'is_unbalance': True,
    'force_col_wise': True,
}
LGBM = LGBMClassifier(**LGBM_PARAMS)

print(f"  Classifier: LightGBM n_estimators={LGBM_PARAMS['n_estimators']} "
      f"max_depth={LGBM_PARAMS['max_depth']} is_unbalance=True native_nan=MIA")


  Classifier: LightGBM n_estimators=150 max_depth=6 is_unbalance=True native_nan=MIA


## CELL AUDIT_RUN -- parquet audit runner (verbatim NB08c_v2 v8)

In [7]:
# @title CELL AUDIT_RUN: parquet audit runner
def audit_parquet(manifest_key, cell_id, pq_id=None):
    '''Run the single audit protocol on one manifest parquet (v8: direction-aware temporal collapse).
    - load parquet
    - if long-format (date column present), aggregate to per-building mean+std
    - select manifest features, drop leakage + count stats + all-NaN
    - fit LightGBM with GroupKFold
    - log to AUDIT_LOG and registry
    Returns: result dict or None (skip).
    '''
    info = MANIFEST['parquets'].get(manifest_key)
    if info is None:
        print(f"\n  [{manifest_key}] SKIP: not in manifest")
        return None
    _pq_id = pq_id or info.get('id', '?')
    print(f"\n  --- [{_pq_id}] {manifest_key} ---")

    try:
        df = load_v2(manifest_key)
    except (FileNotFoundError, KeyError) as e:
        print(f"    SKIP load: {e}")
        return None
    if len(df) == 0:
        print(f"    SKIP: empty dataframe")
        del df; gc.collect()
        return None

    is_long = 'date' in df.columns
    n_rows_raw = len(df)
    n_cities_raw = df['city'].nunique()

    # manifest feature set, cleaned
    feat_cols = get_parquet_features(manifest_key, df)
    if is_long and feat_cols:
        print(f"    Long format: {n_rows_raw:,} rows, {n_cities_raw} cities -> aggregating per-building")
        df = aggregate_long_to_wide(df, feat_cols, manifest_key=manifest_key)
        # re-merge target after aggregation
        df = df.merge(df_bldg[['building_id', 'city', TARGET_COL]].drop_duplicates(),
                      on=['building_id', 'city'], how='inner')
        df = df[df[TARGET_COL] >= 0].copy()
        feat_cols = [c for c in df.columns
                     if c not in ('city', 'building_id', TARGET_COL)]
        feat_cols = drop_leakage(feat_cols)
        feat_cols = [c for c in feat_cols if not _has_count_stat(c)]
        nan_rate = df[feat_cols].isna().mean()
        feat_cols = [c for c in feat_cols if nan_rate[c] < 1.0]

    n_rows = len(df)
    n_cities = df['city'].nunique()
    print(f"    Wide: {n_rows:,} rows, {n_cities} cities, {len(feat_cols)} features")

    if len(feat_cols) < 2:
        print(f"    SKIP: {len(feat_cols)} feature columns after cleaning")
        del df; gc.collect()
        return None

    # overall NaN rate on the post-cleaning feature matrix
    overall_nan = float(df[feat_cols].isna().mean().mean())
    print(f"    Overall NaN rate: {overall_nan*100:.1f}%")

    prep = prepare_features_mia(df, feat_cols)
    if prep is None:
        del df; gc.collect()
        return None
    X, y, groups, building_ids, clean = prep

    exp_name = f"C_{_pq_id}_{manifest_key}"
    res = evaluate_audit(LGBM, X, y, groups, exp_name, building_ids=building_ids)
    if res is not None:
        res['manifest_key'] = manifest_key
        res['parquet_id'] = _pq_id
        res['format'] = 'long->wide' if is_long else 'wide'
        res['overall_nan_pct'] = overall_nan * 100
        res['n_rows_raw'] = n_rows_raw
        res['n_cities_raw'] = n_cities_raw
        log_audit(res, cell_id=cell_id, manifest_key=manifest_key,
                  note=f'{_pq_id} {manifest_key} audit sweep')

    del df, X, y, groups
    gc.collect()
    return res


## CELL BASELINE -- replicate source experiment cell legs

In [8]:
# @title CELL BASELINE: replicate source experiment cell (verbatim audit_parquet calls)
print("=" * 70)
print(f"CELL BASELINE: {NB11_NAME} baseline replication")
print("=" * 70)

NB11_BASELINE_RESULTS = {}
NB11_PREP = {}        # pq_id -> dict(X, y, groups, building_ids, clean_cols, ...)

for pq_id, manifest_key, expected_auc in NB11_PARQUETS:
    print(f"\n  --- [{pq_id}] {manifest_key} (expected AUC={expected_auc:.4f}) ---")
    res = audit_parquet(manifest_key, cell_id=NB11_CELL_ID, pq_id=pq_id)
    assert res is not None, f"audit_parquet returned None for {pq_id}/{manifest_key}"

    got = res['auc_groupkfold']
    delta = got - expected_auc
    ok = abs(delta) <= NB11_BASELINE_TOL
    flag = 'OK' if ok else 'FAIL'
    print(f"    Replicated baseline AUC = {got:.4f}  expected = {expected_auc:.4f}  "
          f"delta = {delta:+.4f}  [{flag}]")
    if not ok:
        raise AssertionError(
            f"Baseline AUC mismatch for {pq_id}/{manifest_key}: got {got:.4f}, "
            f"expected {expected_auc:.4f} (tol {NB11_BASELINE_TOL}). "
            "Do not proceed to Optuna -- investigate prep-chain drift first."
        )

    NB11_BASELINE_RESULTS[pq_id] = res

    # Rebuild X/y/groups/building_ids/clean_cols on the same prep path (matches audit_parquet)
    print(f"    Rebuilding X, y, groups for Optuna (same prep path)...")
    _df = load_v2(manifest_key)
    _is_long = 'date' in _df.columns
    _feat_cols = get_parquet_features(manifest_key, _df)
    if _is_long and _feat_cols:
        _df = aggregate_long_to_wide(_df, _feat_cols, manifest_key=manifest_key)
        _df = _df.merge(df_bldg[['building_id', 'city', TARGET_COL]].drop_duplicates(),
                        on=['building_id', 'city'], how='inner')
        _df = _df[_df[TARGET_COL] >= 0].copy()
        _feat_cols = [c for c in _df.columns
                      if c not in ('city', 'building_id', TARGET_COL)]
        _feat_cols = drop_leakage(_feat_cols)
        _feat_cols = [c for c in _feat_cols if not _has_count_stat(c)]
        _nan_rate = _df[_feat_cols].isna().mean()
        _feat_cols = [c for c in _feat_cols if _nan_rate[c] < 1.0]

    _prep = prepare_features_mia(_df, _feat_cols)
    assert _prep is not None, f"prepare_features_mia returned None for {pq_id}"
    X, y, groups, building_ids, clean_cols = _prep
    NB11_PREP[pq_id] = {
        'X': X, 'y': y, 'groups': groups,
        'building_ids': building_ids, 'clean_cols': clean_cols,
        'manifest_key': manifest_key, 'baseline_auc': got, 'expected_auc': expected_auc,
    }
    print(f"    X shape: {X.shape}  cities: {len(np.unique(groups))}  n_feat: {len(clean_cols)}")

    del _df, _feat_cols
    gc.collect()

print("\n  -> All baselines match source; safe to proceed to Optuna.")


CELL BASELINE: NB11d_A9_A10_composite_prepost baseline replication

  --- [A9] composite_prepost_bands (expected AUC=0.7860) ---

  --- [A9] composite_prepost_bands ---
    Wide: 480,313 rows, 19 cities, 135 features
    Overall NaN rate: 19.4%
    C_A9_composite_prepost_bands             AUC=0.786  F1=0.089  n=480,313  cities=19  nfeat=135
  REG: C_A9_composite_prepost_bands                  AUC=0.7862 F1=0.0893 n=480313 feat=0 cities=19 [NB08c_v2/cell_c3]
    Replicated baseline AUC = 0.7862  expected = 0.7860  delta = +0.0002  [OK]
    Rebuilding X, y, groups for Optuna (same prep path)...
    X shape: (480313, 135)  cities: 19  n_feat: 135

  --- [A10] composite_prepost_landuse (expected AUC=0.7730) ---

  --- [A10] composite_prepost_landuse ---
    Wide: 480,313 rows, 19 cities, 4 features
    Overall NaN rate: 14.6%
    C_A10_composite_prepost_landuse          AUC=0.773  F1=0.083  n=480,313  cities=19  nfeat=4
  REG: C_A10_composite_prepost_landuse               AUC=0.7727 F1=0.0

## CELL OPTUNA -- LightGBM-MIA hyperparameter search per parquet

Search space (10D): `n_estimators [100,800]`, `num_leaves [15,255]`, `max_depth [3,12]`, `learning_rate [0.01,0.2]` log, `min_child_samples [5,100]`, `reg_alpha [0,2]`, `reg_lambda [0,2]`, `feature_fraction [0.5,1]`, `bagging_fraction [0.5,1]`, `bagging_freq [0,10]`. Fixed: `is_unbalance=True`, `force_col_wise=True`, `random_state`, `n_jobs=-1`, `verbose=-1`.

In [9]:
# @title CELL OPTUNA: LightGBM-MIA search per parquet
import optuna
from optuna.samplers import TPESampler
import joblib

NB11_OUT_DIR = OUTPUTS_DIR / "NB11_V2"
(NB11_OUT_DIR / "studies").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "oof").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "best_params").mkdir(parents=True, exist_ok=True)
(NB11_OUT_DIR / "summary").mkdir(parents=True, exist_ok=True)

# LGBM_PARAMS comes from CELL H (verbatim source); fixed for all trials
FIXED_LGBM = dict(
    random_state=RANDOM_STATE,
    n_jobs=NB11_THREADS_PER_NB,
    verbose=-1,
    is_unbalance=True,
    force_col_wise=True,
)

BASELINE_TRIAL_LGBM = {
    'n_estimators': LGBM_PARAMS['n_estimators'],
    'num_leaves': LGBM_PARAMS['num_leaves'],
    'max_depth': LGBM_PARAMS['max_depth'],
    'learning_rate': LGBM_PARAMS['learning_rate'],
    'min_child_samples': LGBM_PARAMS['min_child_samples'],
    'reg_alpha': 0.0, 'reg_lambda': 0.0,
    'feature_fraction': 1.0, 'bagging_fraction': 1.0, 'bagging_freq': 0,
}

def _oof_auc_lgbm(params, X, y, groups):
    gkf = GroupKFold(n_splits=min(N_FOLDS, len(np.unique(groups))))
    y_proba_oof = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups):
        clf = LGBMClassifier(**{**FIXED_LGBM, **params})
        clf.fit(X[tr], y[tr])
        y_proba_oof[te] = clf.predict_proba(X[te])[:, 1]
    valid = ~np.isnan(y_proba_oof)
    return roc_auc_score(y[valid], y_proba_oof[valid])

def suggest_lgbm(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 0, 10),
    }

def patience_stop_factory(patience):
    def cb(study, trial):
        best = study.best_trial
        if (trial.number - best.number) >= patience:
            print(f"      [patience] no improvement for {patience} trials -> stopping")
            study.stop()
    return cb

NB11_OPTUNA_RESULTS = {}      # pq_id -> dict
for pq_id, manifest_key, _expected in NB11_PARQUETS:
    prep = NB11_PREP[pq_id]
    X_pq, y_pq, groups_pq = prep['X'], prep['y'], prep['groups']
    baseline_auc = prep['baseline_auc']
    n_trials = OPTUNA_BASE_BUDGET if baseline_auc >= OPTUNA_LOW_AUC_CUTOFF else OPTUNA_LOW_AUC_BUDGET

    STUDY_TAG = f"{NB11_NAME}__{manifest_key}__LightGBM-MIA"
    STUDY_DB  = NB11_OUT_DIR / "studies" / f"{STUDY_TAG}.sqlite"
    storage_url = f"sqlite:///{STUDY_DB.as_posix()}"

    print(f"\n  === [{pq_id}] {manifest_key} ===")
    print(f"    baseline_auc = {baseline_auc:.4f}   budget = {n_trials} trials   patience = {OPTUNA_PATIENCE}")
    print(f"    DB: {STUDY_DB}")

    def objective(trial, X=X_pq, y=y_pq, g=groups_pq):
        params = suggest_lgbm(trial)
        return _oof_auc_lgbm(params, X, y, g)

    study = optuna.create_study(
        direction=OPTUNA_DIRECTION,
        storage=storage_url,
        study_name=STUDY_TAG,
        load_if_exists=True,
        sampler=TPESampler(seed=OPTUNA_SEED),
    )

    # RESUME-IF-EXISTS: any complete trial -> use as-is, no further optimization.
    # To force re-run, delete the corresponding sqlite file under NB11_V2/studies/.
    completed_trials = [t for t in study.trials if t.state.name == 'COMPLETE']
    if len(completed_trials) > 0:
        best = study.best_trial
        n_trials_done = len(study.trials)
        print(f"    RESUME: {n_trials_done} trials in study ({len(completed_trials)} complete) -- using as-is")
        print(f"    Best AUC: {best.value:.4f}  (trial {best.number})")
        print(f"    Best params: {best.params}")
        print(f"    Delta vs baseline: {best.value - baseline_auc:+.4f}")
    else:
        study.enqueue_trial(BASELINE_TRIAL_LGBM)
        print(f"    enqueued baseline params as trial 0: {BASELINE_TRIAL_LGBM}")
        study.optimize(objective, n_trials=n_trials,
                       callbacks=[patience_stop_factory(OPTUNA_PATIENCE)],
                       show_progress_bar=False)
        n_trials_done = len(study.trials)
        best = study.best_trial
        print(f"    Optuna done: {n_trials_done} trials")
        print(f"    Best AUC: {best.value:.4f}  (trial {best.number})")
        print(f"    Best params: {best.params}")
        print(f"    Delta vs baseline: {best.value - baseline_auc:+.4f}")

    NB11_OPTUNA_RESULTS[pq_id] = {
        'manifest_key': manifest_key,
        'baseline_auc': baseline_auc,
        'tuned_auc': best.value,
        'best_params': best.params,
        'n_trials_done': n_trials_done,
        'n_trials_requested': n_trials,
        'study_tag': STUDY_TAG,
        'study_db': STUDY_DB,
    }



  === [A9] composite_prepost_bands ===
    baseline_auc = 0.7862   budget = 200 trials   patience = 40
    DB: /content/drive_f/masterthesis/data/outputs/NB11_V2/studies/NB11d_A9_A10_composite_prepost__composite_prepost_bands__LightGBM-MIA.sqlite


[I 2026-06-21 17:17:53,777] Using an existing study with name 'NB11d_A9_A10_composite_prepost__composite_prepost_bands__LightGBM-MIA' instead of creating a new one.
[I 2026-06-21 17:17:53,961] Using an existing study with name 'NB11d_A9_A10_composite_prepost__composite_prepost_landuse__LightGBM-MIA' instead of creating a new one.


    RESUME: 177 trials in study (177 complete) -- using as-is
    Best AUC: 0.8182  (trial 136)
    Best params: {'n_estimators': 192, 'num_leaves': 239, 'max_depth': 3, 'learning_rate': 0.133931245425862, 'min_child_samples': 72, 'reg_alpha': 0.5710957972339202, 'reg_lambda': 0.4315572833030301, 'feature_fraction': 0.7443263274151685, 'bagging_fraction': 0.7295844864060694, 'bagging_freq': 3}
    Delta vs baseline: +0.0320

  === [A10] composite_prepost_landuse ===
    baseline_auc = 0.7727   budget = 200 trials   patience = 40
    DB: /content/drive_f/masterthesis/data/outputs/NB11_V2/studies/NB11d_A9_A10_composite_prepost__composite_prepost_landuse__LightGBM-MIA.sqlite
    RESUME: 65 trials in study (65 complete) -- using as-is
    Best AUC: 0.8028  (trial 24)
    Best params: {'n_estimators': 711, 'num_leaves': 92, 'max_depth': 4, 'learning_rate': 0.1253512068224737, 'min_child_samples': 59, 'reg_alpha': 0.9536822441813846, 'reg_lambda': 0.9326770668008164, 'feature_fraction': 0.88

## CELL OPTUNA-FINAL -- refit + save OOF/model/JSON/registry with `mean_folds_auc`

In [10]:
import importlib, nb11_finalize
importlib.reload(nb11_finalize)
nb11_finalize.finalize_group_a(globals())


  === refit [A9] composite_prepost_bands ===
    Tuned OOF AUC (refit): 0.8182  (study.best=0.8182)  delta=-0.0000
    mean(folds)=0.6820  std(folds)=0.0800
    saved oof -> oof_NB11d_A9_A10_composite_prepost__composite_prepost_bands__LightGBM-MIA-Optuna__20260621_171446_115af1.parquet  TP=5680 FN=1506 FP=122338 TN=350789
    Saved model: /content/drive_f/masterthesis/data/outputs/NB11_V2/models/NB11d_A9_A10_composite_prepost__composite_prepost_bands__LightGBM-MIA__best.joblib
    Saved best_params: /content/drive_f/masterthesis/data/outputs/NB11_V2/best_params/NB11d_A9_A10_composite_prepost__composite_prepost_bands__LightGBM-MIA__best.json
  REG: C_A9_composite_prepost_bands__OPTUNA          AUC=0.8182 F1=0.0840 n=480313 feat=135 cities=19 [NB08c_v2/cell_c3]

  === refit [A10] composite_prepost_landuse ===
    Tuned OOF AUC (refit): 0.8028  (study.best=0.8028)  delta=+0.0000
    mean(folds)=0.6689  std(folds)=0.0705
    saved oof -> oof_NB11d_A9_A10_composite_prepost__composite_prepo

## CELL SUMMARY -- multi-row CSV with `mean_folds_auc`/`std_folds_auc` + per-fold AUC print

In [11]:
# @title CELL SUMMARY
rows = []
for pq_id, info in NB11_OPTUNA_RESULTS.items():
    prep = NB11_PREP[pq_id]
    rows.append({
        'notebook': NB11_NAME,
        'parquet_id': pq_id,
        'manifest_key': info['manifest_key'],
        'classifier': 'LightGBM-MIA',
        'baseline_auc': info['baseline_auc'],
        'baseline_auc_expected': prep['expected_auc'],
        'tuned_auc': info['tuned_auc_refit'],
        'delta_auc': info['tuned_auc_refit'] - info['baseline_auc'],
        'mean_folds_auc': info['mean_folds_auc'],
        'std_folds_auc': info['std_folds_auc'],
        'n_trials_requested': info['n_trials_requested'],
        'n_trials_done': info['n_trials_done'],
        'early_stopped': info['n_trials_done'] < info['n_trials_requested'],
        'patience': OPTUNA_PATIENCE,
        'n_features': len(prep['clean_cols']),
        'n_buildings': len(prep['y']),
        'n_cities': int(len(np.unique(prep['groups']))),
        'study_tag': info['study_tag'],
    })
df_summary = pd.DataFrame(rows).sort_values('tuned_auc', ascending=False)
summary_path = NB11_OUT_DIR / "summary" / f"{NB11_NAME}_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(df_summary.to_string(index=False))
print(f"\n  Saved summary: {summary_path}")

print("\n  Per-fold AUC (tuned models):")
for pq_id, info in NB11_OPTUNA_RESULTS.items():
    fold_aucs = info['fold_aucs']
    print(f"    [{pq_id}] {info['manifest_key']:36s} pooled={info['tuned_auc_refit']:.4f}  "
          f"mean(folds)={info['mean_folds_auc']:.4f}  std={info['std_folds_auc']:.4f}  "
          f"folds={[f'{a:.3f}' for a in fold_aucs]}")

print(f"\n  {NB11_NAME} complete.")


                      notebook parquet_id              manifest_key   classifier  baseline_auc  baseline_auc_expected  tuned_auc  delta_auc  mean_folds_auc  std_folds_auc  n_trials_requested  n_trials_done  early_stopped  patience  n_features  n_buildings  n_cities                                                               study_tag
NB11d_A9_A10_composite_prepost         A9   composite_prepost_bands LightGBM-MIA      0.786185                  0.786   0.818172   0.031987        0.681979       0.079955                 200            177           True        40         135       480313        19   NB11d_A9_A10_composite_prepost__composite_prepost_bands__LightGBM-MIA
NB11d_A9_A10_composite_prepost        A10 composite_prepost_landuse LightGBM-MIA      0.772662                  0.773   0.802828   0.030165        0.668888       0.070519                 200             65           True        40           4       480313        19 NB11d_A9_A10_composite_prepost__composite_prepost_landuse_